In [1]:
import pytest
from scipy.stats import norm
import numpy as np
import time
import import_ipynb
from GBM import simulate_GBM, simulate_GBM_antithetic, simulate_GBM_numba_parallel, simulate_GBM_numba
from black_scholes import Black_Scholes, bs_delta, bs_vega
from european import european_mc, european_mc_antithetic, european_mc_control_variate
from exotic import asian_mc, geometric_asian_bs, price_lookback_mc, barrier_mc
from greeks import pathwise_delta, pathwise_vega, lr_delta, lr_vega, fd_delta, fd_vega
from american import american_longstaff_schwartz, american_binomial, american_longstaff_schwartz_fast

import matplotlib.pyplot as plt
from scipy.stats import qmc

import cProfile
import numba
from numba import njit, prange

In [14]:
%%writefile tests/conftest.py
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import numpy as np
np.random.seed(42)

import import_ipynb
import pytest

@pytest.fixture
def params():
    return dict(S0=100, K=100, r=0.05, sigma=0.2, T=1.0)

@pytest.fixture
def bs_call_price():
    return 10.4506

@pytest.fixture
def bs_put_price():
    return 5.5735

Overwriting tests/conftest.py


In [15]:
%%writefile tests/test_black_scholes.py
import numpy as np
from black_scholes import Black_Scholes
import numpy as np
np.random.seed(42)

def test_bs_call_atm(params, bs_call_price):
    price = Black_Scholes(**params, type="call")
    assert abs(price - bs_call_price) < 0.001

def test_bs_off_atm_regression():
    # this exact case caught a real precedence bug earlier in the project - keep permanently
    price = Black_Scholes(100, 110, 0.05, 0.2, 1.0, type="call")
    assert abs(price - 6.0401) < 0.001

def test_bs_put_call_parity(params):
    call = Black_Scholes(**params, type="call")
    put = Black_Scholes(**params, type="put")
    S0, K, r, T = params["S0"], params["K"], params["r"], params["T"]
    assert abs((call - put) - (S0 - K*np.exp(-r*T))) < 1e-6

Overwriting tests/test_black_scholes.py


In [4]:
%%writefile tests/test_european.py
import numpy as np
np.random.seed(42)

from european import european_mc, european_mc_antithetic, european_mc_control_variate, european_sobol_rqmc_error

def test_naive_mc_converges_to_bs(params, bs_call_price):
    price, se = european_mc(**params, n=252, n_paths=200_000, type="call")
    assert abs(price - bs_call_price) < 3 * se

def test_antithetic_converges_to_bs(params, bs_call_price):
    price, se = european_mc_antithetic(**params, n=252, n_paths=200_000, type="call")
    assert abs(price - bs_call_price) < 3 * se

def test_control_variate_converges_to_bs(params, bs_call_price):
    price, se = european_mc_control_variate(**params, n=252, n_paths=200_000, type="call")
    assert abs(price - bs_call_price) < 3 * se

def test_sobol_converges_to_bs(params, bs_call_price):
    price, se = european_sobol_rqmc_error(S0=params["S0"], K=params["K"], r=params["r"],
                                            sigma=params["sigma"], T=params["T"],
                                            n_paths=10_000, type="call")
    assert abs(price - bs_call_price) < 3 * se

def test_antithetic_reduces_variance(params):
    _, se_naive = european_mc(**params, n=252, n_paths=50_000, type="call")
    _, se_anti = european_mc_antithetic(**params, n=252, n_paths=50_000, type="call")
    assert se_anti < se_naive

Overwriting tests/test_european.py


In [16]:
%%writefile tests/test_greeks.py
from greeks import pathwise_delta, pathwise_vega, lr_delta, lr_vega
from black_scholes import bs_delta, bs_vega
import numpy as np
np.random.seed(42)

def test_pathwise_delta_matches_bs(params):
    delta, se = pathwise_delta(**params, n_paths=200_000, type="call")
    bs_d = bs_delta(**params, type="call")
    assert abs(delta - bs_d) < 3 * se

def test_lr_delta_matches_bs(params):
    delta, se = lr_delta(**params, n_paths=200_000, type="call")
    bs_d = bs_delta(**params, type="call")
    assert abs(delta - bs_d) < 3 * se

def test_pathwise_vega_matches_bs(params):
    vega, se = pathwise_vega(**params, n_paths=200_000, type="call")
    bs_v = bs_vega(S0=params["S0"], K=params["K"], r=params["r"], sigma=params["sigma"], T=params["T"])
    assert abs(vega - bs_v) < 3 * se

def test_pathwise_more_precise_than_lr(params):
    _, se_pw = pathwise_delta(**params, n_paths=100_000, type="call")
    _, se_lr = lr_delta(**params, n_paths=100_000, type="call")
    assert se_pw < se_lr

Overwriting tests/test_greeks.py


In [17]:
%%writefile tests/test_exotic.py
from exotic import asian_mc, geometric_asian_bs, price_lookback_mc, barrier_mc
from european import european_mc
import numpy as np
np.random.seed(42)

def test_geometric_asian_matches_closed_form(params):
    price, se = asian_mc(**params, n=252, n_paths=100_000, average_type="geometric", type="call")
    bs_price = geometric_asian_bs(**params, type="call")
    assert abs(price - bs_price) < 3 * se

def test_arithmetic_asian_exceeds_geometric(params):
    arith_price, _ = asian_mc(**params, n=252, n_paths=100_000, average_type="arithmetic", type="call")
    geo_price, _ = asian_mc(**params, n=252, n_paths=100_000, average_type="geometric", type="call")
    assert arith_price > geo_price

def test_lookback_exceeds_vanilla(params):
    lb_price, _ = price_lookback_mc(S0=params["S0"], r=params["r"], sigma=params["sigma"],
                                       T=params["T"], n=252, n_paths=100_000, type="call")
    vanilla_price, _ = european_mc(**params, n=252, n_paths=100_000, type="call")
    assert lb_price > vanilla_price

def test_barrier_in_out_parity(params):
    B = 130
    price_out, se_out = barrier_mc(**params, B=B, n=252, n_paths=100_000, type="call", barrier_type="up-and-out")
    price_in, se_in = barrier_mc(**params, B=B, n=252, n_paths=100_000, type="call", barrier_type="up-and-in")
    vanilla_price, se_vanilla = european_mc(**params, n=252, n_paths=100_000, type="call")
    combined_se = (se_out**2 + se_in**2 + se_vanilla**2) ** 0.5
    assert abs((price_out + price_in) - vanilla_price) < 4 * combined_se

def test_barrier_extreme_out_matches_vanilla(params):
    B_extreme = 1000
    price_out, _ = barrier_mc(**params, B=B_extreme, n=252, n_paths=100_000, type="call", barrier_type="up-and-out")
    vanilla_price, _ = european_mc(**params, n=252, n_paths=100_000, type="call")
    assert abs(price_out - vanilla_price) < 0.15

def test_barrier_extreme_in_near_zero(params):
    B_extreme = 1000
    price_in, _ = barrier_mc(**params, B=B_extreme, n=252, n_paths=100_000, type="call", barrier_type="up-and-in")
    assert price_in < 0.05

Overwriting tests/test_exotic.py


In [19]:
%%writefile tests/test_american.py
from american import american_longstaff_schwartz, american_binomial, american_longstaff_schwartz_fast
from european import european_mc
import numpy as np
np.random.seed(42)

def test_american_put_exceeds_european(params):
    am_price, _ = american_longstaff_schwartz(**params, n=252, n_paths=100_000, type="put")
    eu_price, _ = european_mc(**params, n=252, n_paths=100_000, type="put")
    assert am_price > eu_price

def test_american_call_equals_european(params):
    am_price, se_am = american_longstaff_schwartz(**params, n=252, n_paths=100_000, type="call")
    eu_price, se_eu = european_mc(**params, n=252, n_paths=100_000, type="call")
    combined_se = (se_am**2 + se_eu**2) ** 0.5
    assert abs(am_price - eu_price) < 5 * combined_se

def test_ls_matches_binomial(params):
    ls_price, se = american_longstaff_schwartz(**params, n=252, n_paths=100_000, type="put")
    binom_price = american_binomial(S0=params["S0"], K=params["K"], r=params["r"],
                                       sigma=params["sigma"], T=params["T"], n=1000, type="put")
    assert abs(ls_price - binom_price) < 3 * se

def test_ls_fast_matches_original(params):
    price_orig, se_orig = american_longstaff_schwartz(**params, n=252, n_paths=100_000, type="put")
    price_fast, se_fast = american_longstaff_schwartz_fast(**params, n=252, n_paths=100_000, type="put")
    assert abs(price_orig - price_fast) < 3 * max(se_orig, se_fast)

Overwriting tests/test_american.py


In [20]:
!pytest tests/ -v

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\Users\ritik\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\ritik\OneDrive\Projects\MCOP\MC_Pricer
plugins: anyio-4.10.0
collecting ... collected 22 items

tests/test_american.py::test_american_put_exceeds_european PASSED        [  4%]
tests/test_american.py::test_american_call_equals_european PASSED        [  9%]
tests/test_american.py::test_ls_matches_binomial PASSED                  [ 13%]
tests/test_american.py::test_ls_fast_matches_original PASSED             [ 18%]
tests/test_black_scholes.py::test_bs_call_atm PASSED                     [ 22%]
tests/test_black_scholes.py::test_bs_off_atm_regression PASSED           [ 27%]
tests/test_black_scholes.py::test_bs_put_call_parity PASSED              [ 31%]
tests/test_european.py::test_naive_mc_converges_to_bs PASSED             [ 36%]
tests/test_european.py::test_antithetic_con

In [35]:
!type tests\test_black_scholes.py

def test_bs_call_atm(params, bs_call_price):
    price = Black_Scholes(**params, type="call")
    assert abs(price - bs_call_price) < 0.001

def test_bs_off_atm_regression():
    # this specific case caught a real precedence bug earlier in the project â€”
    # keep it permanently to prevent silent regression
    price = Black_Scholes(100, 110, 0.05, 0.2, 1.0, type="call")
    assert abs(price - 6.0401) < 0.001

def test_bs_put_call_parity(params):
    call = Black_Scholes(**params, type="call")
    put = Black_Scholes(**params, type="put")
    S0, K, r, T = params["S0"], params["K"], params["r"], params["T"]
    assert abs((call - put) - (S0 - K*np.exp(-r*T))) < 1e-6


In [38]:
!type tests\test_black_scholes.py

import numpy as np
from black_scholes import Black_Scholes

def test_bs_call_atm(params, bs_call_price):
    price = Black_Scholes(**params, type="call")
    assert abs(price - bs_call_price) < 0.001

def test_bs_off_atm_regression():
    # this exact case caught a real precedence bug earlier in the project - keep permanently
    price = Black_Scholes(100, 110, 0.05, 0.2, 1.0, type="call")
    assert abs(price - 6.0401) < 0.001

def test_bs_put_call_parity(params):
    call = Black_Scholes(**params, type="call")
    put = Black_Scholes(**params, type="put")
    S0, K, r, T = params["S0"], params["K"], params["r"], params["T"]
    assert abs((call - put) - (S0 - K*np.exp(-r*T))) < 1e-6


In [9]:
!type tests\conftest.py

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import pytest

@pytest.fixture
def params():
    return dict(S0=100, K=100, r=0.05, sigma=0.2, T=1.0)

@pytest.fixture
def bs_call_price():
    return 10.4506

@pytest.fixture
def bs_put_price():
    return 5.5735


In [10]:
!dir *.py

 Volume in drive C is Windows
 Volume Serial Number is 3880-B4CE

 Directory of C:\Users\ritik\OneDrive\Projects\MCOP\MC_Pricer



File Not Found


In [21]:
!jupyter nbconvert --to script european.ipynb --stdout

#!/usr/bin/env python
# coding: utf-8

# In[5]:


from scipy.stats import norm
import numpy as np
import time
#from Geometric_Brownian_Motion import simulate_GBM
import import_ipynb
from GBM import simulate_GBM, simulate_GBM_antithetic
import matplotlib.pyplot as plt
from scipy.stats import qmc


# In[1]:


def european_mc(S0, K, r, sigma, T, n, n_paths, type="call"):
    t, S = simulate_GBM(S0, r,  sigma, T, n, n_paths)
    S_T = S[:,-1]
    if type == "call":
        payoff = np.maximum(S_T-K,0)
    else:
        payoff = np.maximum(K-S_T,0)

    discount = np.exp(-r*T)*payoff
    price = discount.mean()
    std_err = discount.std(ddof=1)/np.sqrt(n_paths)
    return price, std_err


# In[2]:


def european_mc_antithetic(S0, K, r, sigma, T, n, n_paths, type="call"):
    t, S = simulate_GBM_antithetic(S0, r, sigma, T, n, n_paths)
    S_T = S[:,-1]
    half = n_paths//2

    if type == "call":
        payoff = np.maximum(S_T-K, 0)
    else:
        payoff = np.maximum(K-S_T, 0)

    dis

[NbConvertApp] Converting notebook european.ipynb to script


In [12]:
for seed in [1, 2, 3]:
    np.random.seed(seed)
    am_price, se_am = american_longstaff_schwartz(**{'S0':100,'K':100,'r':0.05,'sigma':0.2,'T':1.0}, n=252, n_paths=100_000, type="call")
    eu_price, se_eu = european_mc(**{'S0':100,'K':100,'r':0.05,'sigma':0.2,'T':1.0}, n=252, n_paths=100_000, type="call")
    print(f"seed={seed}: diff={am_price-eu_price:.4f}, combined_se={(se_am**2+se_eu**2)**0.5:.4f}")

seed=1: diff=0.0966, combined_se=0.0654
seed=2: diff=-0.0338, combined_se=0.0656
seed=3: diff=0.0792, combined_se=0.0657
